# Detecting and Handling Outliers

A few extreme values can pull a linear regression far away from the rest of the data. This notebook shows how to find such **outliers**, how a transformation can reduce their influence, and how to measure whether removing them actually helps the model.

## Learning Objectives

At the end of this notebook, you should be able to:

- Detect outliers with histograms, box plots and scatter plots.
- Calculate z-scores and count how many rows a threshold of 3 would remove.
- Apply a log transformation to skewed features, to the training and the test set alike.
- Compare models trained with and without outliers on the same test set.

## What are Outliers?

> Wikipedia Definition:
> In statistics, an outlier is an observation point that is distant from other observations.

**Outliers** are extreme values that deviate from the other observations in the data. They may indicate variability in a measurement, an experimental error or a novelty. In other words, an outlier is an observation that diverges from the overall pattern of a sample.

Outliers enter the data during the first step of a data science project: data collection. At that stage you usually do not know they are there. An outlier can be the result of a mistake during data collection, or simply a sign of the natural variance in your data.

The most common causes of outliers in a dataset are:
- Data entry errors (human errors)
- Measurement errors (instrument errors)
- Experimental errors (data extraction or experiment planning/executing errors)
- Intentional (dummy outliers made to test detection methods)
- Data processing errors (data manipulation or unintended changes to the dataset)
- Sampling errors (extracting or mixing data from wrong or various sources)
- Natural (not an error, novelties in data)

The aim of outlier detection is therefore not only to find outliers but also to find their origin. To do that, you need to understand the dataset and the data collection process.

There are two types of outliers:
- **Univariate outliers** can be found by looking at the distribution of values of a single feature.
- **Multivariate outliers** can only be found in an n-dimensional space (of n features).

In machine learning, and in any quantitative discipline, the quality of the data is as important as the quality of the prediction or classification model.

When you start looking for outliers, you have to answer two questions about your dataset:
- Which and how many features am I taking into account to detect outliers? (**univariate** / **multivariate**)
- Can I assume a distribution of values for my selected features? (**parametric** / **non-parametric**)

In this notebook we only look at univariate parametric outliers. You will learn how to detect multivariate outliers later in the bootcamp. We will use the [`scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) module to detect outliers by calculating the [`z-score`](https://en.wikipedia.org/wiki/Standard_score) of each feature.


## Finding Outliers

### The Visual Way

You can use visualisations to find outliers. Three kinds of plots are especially suited to visualising outliers:
- box plots
- histograms
- scatter plots

### The Mathematical Way

The **z-score** describes each data point by how many standard deviations it lies from the mean of its feature. After the conversion, every feature has mean 0 and standard deviation 1; the shape of the distribution does not change.

The usual rule, "a z-score beyond 3 is an outlier", assumes the feature is roughly normally distributed. Many features are skewed instead, and a transformation such as the logarithm can make them closer to normal first. That is what we will do below.

Why does this help to identify outliers? Calculating the z-score rescales and centres the data, so the outliers are the data points that lie too far from zero.

Libraries such as SciPy and scikit-learn provide easy-to-use functions and classes for this, which work together with pandas and NumPy. After making the appropriate transformations to the selected features of the dataset, the z-score of any data point can be calculated with the following expression:

$$ z = \frac{x - \mu}{\sigma} $$

where:
- $x$ is the value of the feature
- $\mu$ is the mean of the feature
- $\sigma$ is the standard deviation of the feature

When you compute the z-score for each sample in the dataset, you must specify a threshold. Common rule-of-thumb thresholds are 2.5, 3, 3.5 or more standard deviations.

By tagging or removing the data points that lie beyond the threshold, we classify the data into outliers and non-outliers.

Z-score pros:
- It is a very effective method if the values of a feature can be described by a Gaussian (normal) distribution. (Parametric)
- It is easy to implement with pandas and `scipy.stats`.

Z-score cons:
- With many features, almost every row has at least one extreme value by chance, so the share of flagged rows grows quickly. Always check how many rows a threshold would remove before removing them.
- It is not recommended when you cannot assume a parametric distribution.

## Imports and Data

So let's import the libraries and the dataset.

In [ ]:
# Import the Libraries
from scipy.stats import zscore
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Libraries for modelling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    root_mean_squared_error,
)

# Set the RSEED for the train-test-split
RSEED = 42

# Set global variables for plots
plt.style.use("fivethirtyeight")
sns.set_color_codes("bright")
plt.rcParams["figure.figsize"] = (15, 10)

Load the white wine dataset. We drop the `quality` column and use `alcohol` as the target.

In [ ]:
# Load the data
df = pd.read_csv("data/winequality-white.csv", sep=";")
df.drop("quality", axis=1, inplace=True)
# rename the columns
df.columns = df.columns.str.replace(" ", "_")
df.head()

Do the train-test split **before** looking for outliers. We treat the test set as unseen data so that we can evaluate the performance of the model at the end. If we also removed outliers from the test set, the model would most likely perform better on the test set than it will later on **real** new data.

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
# Define features and target
y = df["alcohol"]
X = df.drop("alcohol", axis=1)

In [ ]:
df.columns

In [ ]:
# Split data into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RSEED)

## Dealing with Outliers

The steps of this section at a glance:

<img src="assets/outlier_workflow.png" width="450">

Now look at the distribution of the features in the training set with different plots. The scatter plots need the target variable, so we concatenate it with the features.

In [ ]:
# Join features and target for plotting
df_train = pd.concat([X_train, y_train], axis=1)

Let's start with some histograms.

In [ ]:
# Plot histograms for train data
fig, ax = plt.subplots(4, 3, figsize=(16, 16))
count = 0
for item in X_train.columns.tolist():
    sns.histplot(
        X_train[item], kde=True, ax=ax[int(count / 3)][count % 3], color="#33658A"
    ).set(title=item, xlabel="")
    count += 1
ax.flat[-2].set_visible(False)
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

In [ ]:
# Plot the training data in box plots, like the histograms above
fig, ax = plt.subplots(4, 3, figsize=(16, 24))
count = 0
for item in X_train.columns.tolist():
    # TODO: draw a box plot of X_train[item] with sns.boxplot, on ax[int(count / 3)][count % 3],
    #       and give it the column name as title, like the histograms
    ...
    count += 1
ax.flat[-2].set_visible(False)
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

Next, look at the correlation between the features.

In [ ]:
# Plot correlation matrix
mask = np.triu(df_train.corr())
plt.figure(figsize=(20, 15))
ax = sns.heatmap(round(df_train.corr(), 1), annot=True, mask=mask, cmap="RdBu_r")

Now take a closer look at individual features, starting with `chlorides`. To inspect more features, add cells and apply the same code to them.

In [ ]:
# Plotting one feature with histogram, box plot and scatter plot
fig, ax = plt.subplots(nrows=4, figsize=(16, 16))
count = 0
# Histogram
sns.histplot(X_train["chlorides"], kde=True, ax=ax[0], color="#33658A").set(
    title="Chlorides", xlabel=""
)
# Box plot
sns.boxplot(x=X_train["chlorides"], ax=ax[1], color="#33658A").set(title="Chlorides")
# Scatter plot
sns.scatterplot(x=X_train.chlorides, y=df_train.alcohol, ax=ax[2], color="#33658A").set(
    title="Chlorides"
)

ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

There seem to be outliers at both ends of the distribution. Since the distribution is clearly right-skewed, there are more outliers on the right.

A first step when you encounter outliers is to transform the data. Sometimes a transformation alone is enough to remove them. Common transformations include:
- Log transformation
- Square root transformation
- Power transformation
- Box-Cox transformation

Let's try the log transformation. To decide which features to transform, measure their **skewness**: 0 means a symmetric distribution, and values above about 1 mean a clearly skewed one.

In [ ]:
# Skewness of each training feature, most skewed first
X_train.skew().sort_values(ascending=False).round(2)

`chlorides` (5.14) and `volatile_acidity` (1.64) are clearly skewed, with long right tails, and the log pulls both back towards a symmetric shape. We transform these two.

Three other features look skewed too, but we leave them alone, each for a reason:

- `citric_acid` (1.38) contains zeros, and the logarithm of 0 is undefined.
- `density` (1.18) is skewed by a handful of extreme values rather than a long tail. The log barely changes that, so the z-scores below will catch those values instead.
- `residual_sugar` (1.16) is related to `alcohol` (and to `density`) linearly in its original units. A log transformation would weaken the model: its test $R^2$ would fall from 0.906 to 0.833.

A transformation is a modelling decision, not only a cleaning step: always check what it does to the model.

In [ ]:
# Log-transform the two clearly skewed features where the log helps
X_train.volatile_acidity = X_train.volatile_acidity.apply(lambda x: np.log(x))
X_train.chlorides = X_train.chlorides.apply(lambda x: np.log(x))

Now check whether the transformation removed the outliers by plotting the `chlorides` column again.

In [ ]:
# Plotting one feature with histogram, box plot and scatter plot
fig, ax = plt.subplots(nrows=4, figsize=(16, 16))
count = 0
# Histogram
sns.histplot(X_train["chlorides"], kde=True, ax=ax[0], color="#33658A").set(
    title="Chlorides", xlabel=""
)
# Box plot
sns.boxplot(x=X_train["chlorides"], ax=ax[1], color="#33658A").set(title="Chlorides")
# Scatter plot
sns.scatterplot(x=X_train.chlorides, y=df_train.alcohol, ax=ax[2], color="#33658A").set(
    title="Chlorides"
)

ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

The skewness is gone, but there are still outliers. So let's calculate the z-score of the features after the transformation. SciPy's `zscore` function calculates the z-score of every column in the dataset at once:

Documentation: [`zscore`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.zscore.html).

In [ ]:
# Rebuild the training DataFrame so it holds the log-transformed features
df_train = pd.concat([X_train, y_train], axis=1)

# Calculate z-score for the training data
df_zscore = df_train.apply(zscore)
df_zscore

Before removing outliers based on the z-score or any other threshold, calculate what percentage of the data you would lose. If the proportion is too large, reconsider your approach.

So let's calculate how many rows we would delete if we removed the outliers with a threshold of 3 standard deviations.

In [ ]:
# Calculate data loss based on 3 standard deviations
num_outlier = len(df_train[(df_train.apply(zscore).abs() > 3).any(axis=1)])
print(f"{num_outlier} rows of {len(df_train)} total rows contain outliers.")
print(f"Percentage of data loss: {num_outlier / len(df_train) * 100:.2f}%")

Roughly 7% of the observations in our DataFrame (260 of 3,673 rows) contain one or more outliers. 7% is quite a lot, especially for a small dataset. Nevertheless, we continue and create a DataFrame without those observations: we calculate the z-score for each column, including the target `alcohol`, take the absolute value, and keep only the rows where every z-score is lower than 3.

In [ ]:
# Calculate z-score and query dataframe to remove rows with z-scores > 3
X_train_wo_outlier = X_train[
    (df_train.select_dtypes(exclude="object").apply(zscore).abs() < 3).all(axis=1)
]

This removed the rows from the feature DataFrame. Now we have to drop the corresponding entries in the target variable as well.

To drop the same rows in `y_train`, we merge `X_train_wo_outlier` and `y_train` on their index and then split them again.

In [ ]:
# Remove outliers in target variable
df_train_wo_outlier = pd.merge(
    X_train_wo_outlier, y_train, left_index=True, right_index=True
)
y_train_wo_outlier = df_train_wo_outlier["alcohol"]
X_train_wo_outlier = df_train_wo_outlier.drop("alcohol", axis=1)

The outliers are now removed from the training data. Let's see what the distributions look like.

In [ ]:
# Plot histograms for cleaned features
fig, ax = plt.subplots(4, 3, figsize=(16, 16))
count = 0
for item in X_train_wo_outlier.columns.tolist():
    sns.histplot(
        X_train_wo_outlier[item],
        kde=True,
        ax=ax[int(count / 3)][count % 3],
        color="#33658A",
    ).set(title=item, xlabel="")
    count += 1
ax.flat[-1].set_visible(False)
ax.flat[-2].set_visible(False)
fig.tight_layout(pad=3)

In [ ]:
# Plot the training data without outliers in box plots, like the histograms above
fig, ax = plt.subplots(4, 3, figsize=(16, 24))
count = 0
for item in X_train_wo_outlier.columns.tolist():
    # TODO: draw a box plot of X_train_wo_outlier[item] with sns.boxplot, on ax[int(count / 3)][count % 3],
    #       and give it the column name as title, like the histograms
    ...
    count += 1
ax.flat[-2].set_visible(False)
ax.flat[-1].set_visible(False)
fig.tight_layout(pad=3)

Did the correlation matrix change as well? Let's look at it again.

In [ ]:
# Plot correlation matrix for cleaned data
mask = np.triu(X_train_wo_outlier.corr())
plt.figure(figsize=(20, 15))
ax = sns.heatmap(
    round(X_train_wo_outlier.corr(), 1), annot=True, mask=mask, cmap="RdBu_r"
)

## Influence of Outlier Removal on Model Performance



Now compare two models:
- a linear regression model trained on the data **with** outliers
- a linear regression model trained on the data **without** outliers

Before we can use the test set to evaluate the models, we need to transform it in the same way as the training set.

In [ ]:
# Apply the same log transformation to the test set: the two features transformed above
X_test.volatile_acidity = X_test.volatile_acidity.apply(lambda x: np.log(x))
# TODO: the other feature: chlorides
...

### Model with Outliers

The model trained on the data with outliers uses `X_train` and `y_train`.

Documentation: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
# Instantiate our model
lin_reg = LinearRegression()

# Fit the model using our train data
lin_reg.fit(X_train, y_train)

# Make predictions on the train data  and on the test data
y_pred_train = lin_reg.predict(X_train)
y_pred = lin_reg.predict(X_test)

### Model without Outliers

This model is trained on the cleaned data, `X_train_wo_outlier` and `y_train_wo_outlier`.

In [ ]:
# TODO: create a LinearRegression
lin_reg_wo_outliers = ...

# TODO: fit it on the training data without outliers: X_train_wo_outlier and y_train_wo_outlier
...

# TODO: predict on that training data, and on the (unchanged) test data
y_pred_train_wo_outlier = ...
y_pred_test_wo_outlier = ...

### Model Comparison

Compare the scores of the two models on the training and the test set:

Documentation: [`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html), [`r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [ ]:
print("With outliers:")
print("R-squared train: ", round(r2_score(y_train, y_pred_train), 3))
print("R-squared test: ", round(r2_score(y_test, y_pred), 3))
print("RMSE train: ", round(root_mean_squared_error(y_train, y_pred_train), 3))
print("RMSE test: ", round(root_mean_squared_error(y_test, y_pred), 3))
print("-----------------------------------")
print("Without outliers:")
print(
    "R-squared train: ", round(r2_score(y_train_wo_outlier, y_pred_train_wo_outlier), 3)
)
print("R-squared test: ", round(r2_score(y_test, y_pred_test_wo_outlier), 3))
print(
    "RMSE train: ",
    round(root_mean_squared_error(y_train_wo_outlier, y_pred_train_wo_outlier), 3),
)
print("RMSE test: ", round(root_mean_squared_error(y_test, y_pred_test_wo_outlier), 3))

Removing the outliers improves the test RMSE from 0.378 to 0.366, and the test $R^2$ from 0.906 to 0.912. That is a **small** gain, and it costs 260 rows (7%) of the training data.

Be careful with the training scores. They improve much more (RMSE from 0.466 to 0.350), but mostly because the hardest rows are no longer in the data being scored, not because the model got better. Only the test set, which still contains its outliers, measures a real improvement.

## Final Remark



Use the method in this notebook with care. We showed how to remove outliers with the z-score method, but we did not look at the business question: does it make sense to remove all the outliers, or are some of them legitimate special cases?

Lose as little data as possible. When you remove rows from the dataset, the model can no longer learn anything about those cases. Remove outliers only if it makes sense from both a statistical and a business point of view.

In some cases you can also remove outliers from the target variable. Do this only if it makes sense from a business perspective, and monitor the variable in production to check that such outliers stay rare and that the distribution of the data does not change. If your model does not work for all customers, you may lose some of them.

## Summary

In this notebook you:

- Detected outliers with histograms, box plots and scatter plots on the training set only.
- Calculated z-scores and counted the rows a threshold of 3 would remove: 260 of 3,673 (7%).
- Measured the skewness of each feature, applied a log transformation to the two features where it helps, and applied the same transformation to the test set.
- Compared models trained with and without outliers on the same test set: removing them improved the test RMSE only slightly, from 0.378 to 0.366.

## References & Further Reading

- [**Outliers: To Drop or Not to Drop**](https://www.theanalysisfactor.com/outliers-to-drop-or-not-to-drop/): practical guidance on when removing outliers is justified.